# AgiBotWorld-Beta Dataset

This notebook demonstrates how to load and sample from the [AgiBotWorld-Beta](https://huggingface.co/datasets/agibot-world/AgiBotWorld-Beta) dataset using `AgiBotWorldBetaDataset`.

The dataset is structured as **one HuggingFace config per task**.  Only the tasks and episodes you request are downloaded and converted — everything else is skipped.

In [1]:
import numpy as np
from robotdataset import AgiBotWorldBetaDataset, list_agibot_tasks, TemporalSampler, batchViz, itemViz

/home/zeus/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1779142525.524424   45901 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779142525.576548   45901 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779142526.798298   45901 port.cc:153] oneDNN custom operations are on. You may se

## 1. Discover available tasks

`list_agibot_tasks()` is a metadata-only call — it does **not** download any data.

In [3]:
tasks = list_agibot_tasks()
print(f"{len(tasks)} tasks available")
tasks[:10]

217 tasks available


[327, 351, 352, 354, 356, 357, 358, 359, 360, 361]

## 2. Load a dataset

Pass one or more task config names.  Only the HuggingFace configs you list are downloaded.

`episodes` filters to specific global episode IDs (assigned deterministically across tasks in sorted order).  Omit `episodes` to load all episodes from the selected tasks.

In [7]:
dataset = AgiBotWorldBetaDataset(
    tasks=[tasks[1]],   # download only the first task
    split="train",
    # episodes=[0, 1],    # convert only episodes 0 and 1
    batch_size=8,
    control_frequency=25.0,
)

print(f"Episodes loaded : {dataset.num_episodes}")
print(f"Total steps     : {len(dataset)}")

KeyboardInterrupt: 

## 3. Inspect available modalities

In [ ]:
for path, spec in dataset.get_modalities().items():
    print(f"{path:45s}  kind={spec['kind']:8s}  dtype={spec.get('dtype','?'):10s}  shape={spec.get('shape','?')}")

## 4. Episode map — global ID → (task, local ID)

In [ ]:
for gid, (task, local_id) in sorted(dataset.episode_map.items()):
    print(f"global {gid:3d}  ←  task={task!r}  local_ep={local_id}")

## 5. Sample a batch (default T=1 for all modalities)

In [ ]:
batch = dataset.sample()
print(batch)

In [ ]:
# Observation image is returned channels-first (B, T, C, H, W)
print("observation/image shape :", batch["observation", "image"].shape)
print("action shape            :", batch["action"].shape)
print("reward shape            :", batch["next", "reward"].shape)

## 6. Temporal sampling — multi-frame windows

`delta_timestamps` maps each modality to a list of time offsets in seconds.  Here we ask for a 0.5 s look-back on the image stream and the current + next step for actions.

In [ ]:
dataset2 = AgiBotWorldBetaDataset(
    tasks=[tasks[0]],
    split="train",
    episodes=[0, 1],
    batch_size=4,
    control_frequency=25.0,
    delta_timestamps={
        "observation/image": np.arange(-0.5, 0.04, 1 / 25).tolist(),  # 13 frames
        "action": [0.0, 1 / 25],                                        # current + next
    },
)

batch2 = dataset2.sample()
print("observation/image shape :", batch2["observation", "image"].shape)  # (B, T, C, H, W)
print("action shape            :", batch2["action"].shape)                # (B, 2, action_dim)

## 7. Visualise a batch of images

In [ ]:
# batchViz expects (B, T, C, H, W) — shows the anchor frame for each item in the batch
batchViz(batch2["observation", "image"])

## 8. Visualise one episode item as a video

In [ ]:
# itemViz plays the T-frame window for batch item 0 as a video
itemViz(batch2["observation", "image"], item_idx=0, is_output_video=True)

## 9. Swap the sampler without rebuilding

`set_sampler` replaces the temporal sampler on the fly — no need to re-download or re-convert episodes.

In [ ]:
new_sampler = TemporalSampler(
    delta_timestamps={
        "observation/image": [0.0],
        "action": np.arange(0.0, 0.5, 1 / 25).tolist(),  # predict next 12 steps
    },
    control_frequency=25.0,
    image_keys=dataset2.image_keys,
)

dataset2.set_sampler(new_sampler)
batch3 = dataset2.sample()
print("action chunk shape:", batch3["action"].shape)  # (B, 12, action_dim)

## 10. Multi-task loading

Pass multiple task names.  Global episode IDs are assigned deterministically: sorted task names, then sorted local episode IDs within each task.

In [ ]:
if len(tasks) >= 2:
    multi_ds = AgiBotWorldBetaDataset(
        tasks=tasks[:2],
        split="train",
        episodes=[0, 1, 2, 3],   # first 4 global episodes across both tasks
        batch_size=8,
        control_frequency=25.0,
    )
    print(f"Tasks   : {multi_ds.tasks}")
    print(f"Episodes: {multi_ds.num_episodes}")
    print(f"Steps   : {len(multi_ds)}")
    for gid, (task, local_id) in sorted(multi_ds.episode_map.items()):
        print(f"  global {gid}  ←  {task}  local_ep={local_id}")

## 11. Iterate as a training dataloader

In [ ]:
N_BATCHES = 5
for i, batch in enumerate(dataset):
    if i >= N_BATCHES:
        break
    obs_img = batch["observation", "image"]
    print(f"batch {i}: image {tuple(obs_img.shape)}  dtype={obs_img.dtype}")